### 3 Approaches to Tokenisation 

#### Approach 1: Word-level Tokenisation 
* Split on spaces and punctuations
* Cons: Requires massive vocabulary to cover every possible word & miss a word, it is tokenised as [UNK]

#### Approach 2: Character-level Tokenisation 
* Split words at every character
* Pros: Vocab is very small 
* Cons: Sequences are extremely long, 10 words becomes ~ 50 character level tokens
* Cons 2: Model needs to learn that 't' 'h' 'e' means 'the' 

#### Approach 3: Subword Tokenisation
* Sweet spot to decompose words into menaningful and re-usable pieces 
* unhappiness = 'un', 'happi', 'ness' 
* Pros: Vocab stays managable as compared to word-level tokenisation & minimal unknown tokens as most words can be built from the subwords present in the dictionary 

### Byte-pair Encoding
Start with individual characters. Count every adjacent pair in the training corpus. Merge the most frequent pair into a new token. Repeat until you reach the target vocabulary size.

### Implementation of BPE

In [1]:
from collections import Counter

In [72]:
class BPETokenizer:
    def __init__(self):
        self.merges = {}
        self.vocab = {}
    
    def _get_pairs(self, tokens):
        pairs = Counter()
        for i in range(len(tokens) - 1):
            pairs[(tokens[i], tokens[i + 1])] += 1
        return pairs
    
    def _merge_pair(self, tokens, pair, new_token):
        merged = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == pair[0] and tokens[i + 1] == pair[1]:
                merged.append(new_token)
                i += 2
            else:
                merged.append(tokens[i])
                i += 1
        return merged
    
    def train(self, text, num_merges):
        tokens = list(text.encode("utf-8"))
        print(tokens)
        self.vocab = {i: bytes([i]) for i in range(256)}

        for i in range(num_merges):
            # print(f"Current iteration of: {i}")
            pairs = self._get_pairs(tokens)
            # print(f"Current pairs = {pairs}")
            if not pairs:
                break
            best_pair = max(pairs, key = pairs.get)
            # print(f"Best pair = {best_pair}")
            new_token = 256 + i
            tokens = self._merge_pair(tokens, best_pair, new_token)
            self.merges[best_pair] = new_token
            # print(f"{self.vocab[best_pair[0]]}, {self.vocab[best_pair[1]]}")
            self.vocab[new_token] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]
            if i == num_merges - 1:
                print(f"Vocab list = {self.vocab}")

        return self
    
    def encode(self, text):
        tokens = list(text.encode('utf-8'))
        for pair, new_token in self.merges.items():
            tokens = self._merge_pair(tokens, pair, new_token)
        return tokens
    
    def decode(self, tokens):
        byte_sequence = b"".join(self.vocab[t] for t in tokens)
        return byte_sequence.decode('utf-8', errors='replace')

In [73]:
corpus = (
    "The cat sat on the mat. The cat ate the rat. "
    "The dog sat on the log. The dog ate the frog. "
    "Natural language processing is the study of how computers "
    "understand and generate human language. "
    "Tokenization is the first step in any NLP pipeline."
)

tokenizer = BPETokenizer()
tokenizer.train(corpus, num_merges=40)

test_sentences = [
    "The cat sat on the mat.",
    "Natural language processing",
    "tokenization pipeline",
    "unhappiness",
]

for sentence in test_sentences:
    encoded = tokenizer.encode(sentence)
    decoded = tokenizer.decode(encoded)
    raw_bytes = len(sentence.encode("utf-8"))
    ratio = len(encoded) / raw_bytes
    print(f"'{sentence}'")
    print(f"  Tokens: {len(encoded)} (from {raw_bytes} bytes) -- ratio: {ratio:.2f}")
    print(f"  Roundtrip: {'PASS' if decoded == sentence else 'FAIL'}")

[84, 104, 101, 32, 99, 97, 116, 32, 115, 97, 116, 32, 111, 110, 32, 116, 104, 101, 32, 109, 97, 116, 46, 32, 84, 104, 101, 32, 99, 97, 116, 32, 97, 116, 101, 32, 116, 104, 101, 32, 114, 97, 116, 46, 32, 84, 104, 101, 32, 100, 111, 103, 32, 115, 97, 116, 32, 111, 110, 32, 116, 104, 101, 32, 108, 111, 103, 46, 32, 84, 104, 101, 32, 100, 111, 103, 32, 97, 116, 101, 32, 116, 104, 101, 32, 102, 114, 111, 103, 46, 32, 78, 97, 116, 117, 114, 97, 108, 32, 108, 97, 110, 103, 117, 97, 103, 101, 32, 112, 114, 111, 99, 101, 115, 115, 105, 110, 103, 32, 105, 115, 32, 116, 104, 101, 32, 115, 116, 117, 100, 121, 32, 111, 102, 32, 104, 111, 119, 32, 99, 111, 109, 112, 117, 116, 101, 114, 115, 32, 117, 110, 100, 101, 114, 115, 116, 97, 110, 100, 32, 97, 110, 100, 32, 103, 101, 110, 101, 114, 97, 116, 101, 32, 104, 117, 109, 97, 110, 32, 108, 97, 110, 103, 117, 97, 103, 101, 46, 32, 84, 111, 107, 101, 110, 105, 122, 97, 116, 105, 111, 110, 32, 105, 115, 32, 116, 104, 101, 32, 102, 105, 114, 115, 116, 32

In [36]:
corpus

'The cat sat on the mat. The cat ate the rat. The dog sat on the log. The dog ate the frog. Natural language processing is the study of how computers understand and generate human language. Tokenization is the first step in any NLP pipeline.'

In [58]:
best_pair = max(pairs, key = pairs.get)

In [62]:
new_token = 256 + 1
merged = []
i = 0
while i < len(corpus):
    if i < len(corpus) - 1 and corpus[i] == best_pair[0] and corpus[i + 1] == best_pair[1]:
        merged.append(new_token)
        i += 2
    else:
        merged.append(corpus[i])
        i += 1

In [67]:
vocab = {i: bytes([i]) for i in range(256)}

In [66]:
best_pair

('e', ' ')

In [63]:
merged

['T',
 'h',
 257,
 'c',
 'a',
 't',
 ' ',
 's',
 'a',
 't',
 ' ',
 'o',
 'n',
 ' ',
 't',
 'h',
 257,
 'm',
 'a',
 't',
 '.',
 ' ',
 'T',
 'h',
 257,
 'c',
 'a',
 't',
 ' ',
 'a',
 't',
 257,
 't',
 'h',
 257,
 'r',
 'a',
 't',
 '.',
 ' ',
 'T',
 'h',
 257,
 'd',
 'o',
 'g',
 ' ',
 's',
 'a',
 't',
 ' ',
 'o',
 'n',
 ' ',
 't',
 'h',
 257,
 'l',
 'o',
 'g',
 '.',
 ' ',
 'T',
 'h',
 257,
 'd',
 'o',
 'g',
 ' ',
 'a',
 't',
 257,
 't',
 'h',
 257,
 'f',
 'r',
 'o',
 'g',
 '.',
 ' ',
 'N',
 'a',
 't',
 'u',
 'r',
 'a',
 'l',
 ' ',
 'l',
 'a',
 'n',
 'g',
 'u',
 'a',
 'g',
 257,
 'p',
 'r',
 'o',
 'c',
 'e',
 's',
 's',
 'i',
 'n',
 'g',
 ' ',
 'i',
 's',
 ' ',
 't',
 'h',
 257,
 's',
 't',
 'u',
 'd',
 'y',
 ' ',
 'o',
 'f',
 ' ',
 'h',
 'o',
 'w',
 ' ',
 'c',
 'o',
 'm',
 'p',
 'u',
 't',
 'e',
 'r',
 's',
 ' ',
 'u',
 'n',
 'd',
 'e',
 'r',
 's',
 't',
 'a',
 'n',
 'd',
 ' ',
 'a',
 'n',
 'd',
 ' ',
 'g',
 'e',
 'n',
 'e',
 'r',
 'a',
 't',
 257,
 'h',
 'u',
 'm',
 'a',
 'n',
 ' ',
 'l'